In [1]:
import json
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import rcParams
from itertools import product, combinations
import pandas as pd

In [2]:
embeddings = ["_alphaearth", "_anysat", "_terramind"]
names = ["AE", "AS", "TM"]
possible_suffixes = ["", "_NULL_RS", "_NULL_DSM", "_NULL_GIU", "_noPOI", "_NULL"]

total_embeddings = []
total_names = []

for r in range(1, len(embeddings) + 1):
    for emb_combo, name_combo in zip(combinations(embeddings, r), combinations(names, r)):
        total_embeddings.append("".join(emb_combo))
        total_names.append("+".join(name_combo))

# total_ckpts = ["ckpt_livability" + e + s for e, s in product(total_embeddings, possible_suffixes)]

In [3]:
# City index setup — run once before city-level sections below
def _load_jsonl_ids(path):
    ids = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                ids.append(json.loads(line)['id'])
    return ids

_test_ids = _load_jsonl_ids('data_livability/json/Livability_test_0320.json')
_id_to_idx = {id_: i for i, id_ in enumerate(_test_ids)}

cities = ['Eindhoven', 'Hengelo', 'Dordrecht', 'Beesel']
city_indices = {}
for city in cities:
    cids = _load_jsonl_ids(f'data_livability/json/Livability_test_{city}_0320.json')
    city_indices[city] = np.array([_id_to_idx[i] for i in cids])

def _city_rmse(preds_path, tgts_path, indices):
    preds = np.load(preds_path)
    tgts = np.load(tgts_path)
    p, t = preds[indices], tgts[indices]
    return np.sqrt(np.mean((p - t) ** 2, axis=0))

print('City sample counts:', {c: len(idx) for c, idx in city_indices.items()})

%run tex_utils

import re as _re

def _count_poi(text):
    text = (text or '').strip()
    if not text or text.lower() == 'null':
        return 0
    return len([e for e in _re.split(r'  +', text) if e.strip()])

_poi_counts = []
with open('data_livability/json/Livability_test_0320.json') as _f:
    for _line in _f:
        _line = _line.strip()
        if _line:
            _poi_counts.append(_count_poi(json.loads(_line).get('text', '')))
_poi_counts = np.array(_poi_counts)

poi_strata_indices = {
    'none':   np.where(_poi_counts == 0)[0],
    'any':    np.where(_poi_counts >= 1)[0],
    'sparse': np.where(_poi_counts == 1)[0],
    'low':    np.where((_poi_counts >= 2) & (_poi_counts <= 3))[0],
    'medium': np.where((_poi_counts >= 4) & (_poi_counts <= 8))[0],
    'dense':  np.where(_poi_counts > 8)[0],
}
print('POI strata counts:', {k: len(v) for k, v in poi_strata_indices.items()})


City sample counts: {'Eindhoven': 6465, 'Hengelo': 3051, 'Dordrecht': 3496, 'Beesel': 428}
POI strata counts: {'none': 1721, 'any': 11719, 'sparse': 1859, 'low': 3181, 'medium': 3767, 'dense': 2912}


# Adding Embeddings to Input

In [4]:
metrics = ['rmse_lbm', 'rmse_tgts_fys', 'rmse_tgts_onv', 'rmse_tgts_soc', 'rmse_tgts_vrz', 'rmse_tgts_won']
metric_labels = ["LIV", "PHY", "NUI", "SOC", "AME", "HOU"]

embedding_ckpts = [""] + total_embeddings
embedding_names = ["BASE"] + total_names

rows = []
for name, e in zip(embedding_names, embedding_ckpts):
    ckpt = "ckpt_livability" + e
    pattern = f"{ckpt}/Livability_test_0320_eval_results_test_final.txt"
    files = glob.glob(pattern)
    print(name, ckpt, files)
    if len(files) == 0:
        continue
    with open(files[0]) as fp:
        result = json.load(fp)
    row = {"model": name}
    row.update({label: result.get(m, np.nan) for m, label in zip(metrics, metric_labels)})
    rows.append(row)

all_metrics = pd.DataFrame(rows).set_index("model")
save_tex(all_metrics, "tex_tables/s1_add_embeddings.tex",
         "RMSE per model across livability domain scores. Bold: lowest per column.",
         "tab:add_embeddings")
all_metrics

BASE ckpt_livability ['ckpt_livability/Livability_test_0320_eval_results_test_final.txt']
AE ckpt_livability_alphaearth ['ckpt_livability_alphaearth/Livability_test_0320_eval_results_test_final.txt']
AS ckpt_livability_anysat ['ckpt_livability_anysat/Livability_test_0320_eval_results_test_final.txt']
TM ckpt_livability_terramind ['ckpt_livability_terramind/Livability_test_0320_eval_results_test_final.txt']
AE+AS ckpt_livability_alphaearth_anysat ['ckpt_livability_alphaearth_anysat/Livability_test_0320_eval_results_test_final.txt']
AE+TM ckpt_livability_alphaearth_terramind ['ckpt_livability_alphaearth_terramind/Livability_test_0320_eval_results_test_final.txt']
AS+TM ckpt_livability_anysat_terramind ['ckpt_livability_anysat_terramind/Livability_test_0320_eval_results_test_final.txt']
AE+AS+TM ckpt_livability_alphaearth_anysat_terramind ['ckpt_livability_alphaearth_anysat_terramind/Livability_test_0320_eval_results_test_final.txt']
Saved: tex_tables/s1_add_embeddings.tex


,LIV,PHY,NUI,SOC,AME,HOU
model,,,,,,
BASE,0.103618,0.026348,0.060564,0.031340,0.053943,0.034138
AE,0.097648,0.024087,0.058996,0.029959,0.047608,0.032804
AS,0.103740,0.027081,0.061776,0.031384,0.051939,0.034077
TM,0.101561,0.025863,0.060257,0.030999,0.052250,0.033559
AE+AS,0.100680,0.024387,0.060010,0.029801,0.043420,0.033535
AE+TM,0.097654,0.024546,0.058524,0.030474,0.045161,0.032778
AS+TM,0.102474,0.026259,0.061499,0.031637,0.051367,0.033665
AE+AS+TM,0.098873,0.023741,0.059861,0.029755,0.044011,0.033792


## City-level results

In [5]:
metrics = ['rmse_lbm', 'rmse_tgts_fys', 'rmse_tgts_onv', 'rmse_tgts_soc', 'rmse_tgts_vrz', 'rmse_tgts_won']
metric_labels = ["LIV", "PHY", "NUI", "SOC", "AME", "HOU"]

rows = []
for city, idx in city_indices.items():
    for name, e in zip(embedding_names, embedding_ckpts):
        ckpt = "ckpt_livability" + e
        preds_f = glob.glob(f"{ckpt}/Livability_test_0320_preds_test_final.npy")
        tgts_f  = glob.glob(f"{ckpt}/Livability_test_0320_tgts_test_final.npy")
        if not preds_f or not tgts_f:
            continue
        rmse = _city_rmse(preds_f[0], tgts_f[0], idx)
        row = {"city": city, "model": name}
        row.update({lbl: rmse[i] for i, lbl in enumerate(metric_labels)})
        rows.append(row)

df = pd.DataFrame(rows).set_index(["city", "model"])
save_tex(df, "tex_tables/s1_city.tex",
         "City-level RMSE per model. Bold: lowest per column within each city.",
         "tab:add_embeddings_city")
df

Saved: tex_tables/s1_city.tex


LIV       PHY       NUI       SOC       AME       HOU
city      model                                                               
Eindhoven BASE      0.093208  0.023899  0.058990  0.029418  0.053324  0.027962
          AE        0.088731  0.022246  0.056734  0.028522  0.045957  0.027361
          AS        0.095832  0.025570  0.061750  0.030160  0.052094  0.028583
          TM        0.092270  0.023535  0.058531  0.029640  0.050183  0.027872
          AE+AS     0.092279  0.022757  0.057295  0.028492  0.042071  0.028464
          AE+TM     0.089172  0.022582  0.055389  0.028368  0.042530  0.028033
          AS+TM     0.093464  0.025079  0.060196  0.030146  0.050789  0.027856
          AE+AS+TM  0.090664  0.022396  0.057705  0.027853  0.041804  0.029054
Hengelo   BASE      0.118897  0.028160  0.055454  0.032689  0.052958  0.040457
          AE        0.108343  0.025736  0.055988  0.029871  0.046198  0.040021
          AS        0.112171  0.028337  0.058811  0.032008  0.050027  0.039418
          TM        0.114090  0.027229  0.059709  0.032508  0.049786  0.040693
          AE+AS     0.110528  0.025596  0.058389  0.029891  0.042439  0.040488
          AE+TM     0.109541  0.025671  0.058046  0.030201  0.044193  0.039672
          AS+TM     0.112678  0.027524  0.060765  0.032583  0.049542  0.039401
          AE+AS+TM  0.111360  0.025239  0.059635  0.031426  0.042999  0.039712
Dordrecht BASE      0.110058  0.028189  0.069119  0.033835  0.057496  0.039303
          AE        0.104397  0.025177  0.066633  0.032903  0.052703  0.035820
          AS        0.111825  0.027679  0.065939  0.033363  0.054946  0.039138
          TM        0.109075  0.027452  0.066093  0.032597  0.059652  0.037352
          AE+AS     0.109049  0.025802  0.067975  0.032543  0.047781  0.036583
          AE+TM     0.103509  0.026360  0.065998  0.034829  0.051469  0.035449
          AS+TM     0.110725  0.025991  0.066223  0.034059  0.055493  0.038778
          AE+AS+TM  0.104555  0.024140  0.065909  0.032164  0.049751  0.037453
Beesel    BASE      0.078827  0.032066  0.040924  0.028322  0.037813  0.022373
          AE        0.088284  0.029067  0.045422  0.026334  0.036990  0.024188
          AS        0.085977  0.034212  0.045507  0.028191  0.034876  0.023172
          TM        0.072993  0.034589  0.034283  0.026312  0.030865  0.020729
          AE+AS     0.073874  0.027279  0.038109  0.024632  0.031285  0.021447
          AE+TM     0.079349  0.029037  0.040178  0.024395  0.034097  0.019593
          AS+TM     0.084568  0.035139  0.043482  0.025702  0.035246  0.022768
          AE+AS+TM  0.071573  0.028670  0.037287  0.024354  0.032064  0.020213

## POI-density stratified results

In [6]:
metrics = ['rmse_lbm', 'rmse_tgts_fys', 'rmse_tgts_onv', 'rmse_tgts_soc', 'rmse_tgts_vrz', 'rmse_tgts_won']
metric_labels = ["LIV", "PHY", "NUI", "SOC", "AME", "HOU"]

rows = []
for stratum, idx in poi_strata_indices.items():
    for name, e in zip(embedding_names, embedding_ckpts):
        ckpt = "ckpt_livability" + e
        preds_f = glob.glob(f"{ckpt}/Livability_test_0320_preds_test_final.npy")
        tgts_f  = glob.glob(f"{ckpt}/Livability_test_0320_tgts_test_final.npy")
        if not preds_f or not tgts_f:
            continue
        rmse = _city_rmse(preds_f[0], tgts_f[0], idx)
        row = {"stratum": stratum, "model": name}
        row.update({lbl: rmse[i] for i, lbl in enumerate(metric_labels)})
        rows.append(row)

df = pd.DataFrame(rows).set_index(["stratum", "model"])
save_tex(df, "tex_tables/s1_poi.tex",
         "POI-density stratified RMSE per model. Bold: lowest per column within each stratum.",
         "tab:add_embeddings_poi")
df

Saved: tex_tables/s1_poi.tex


LIV       PHY       NUI       SOC       AME       HOU
stratum model                                                               
none    BASE      0.092039  0.031603  0.050337  0.034073  0.048769  0.031181
        AE        0.087955  0.027278  0.049982  0.031918  0.042125  0.031012
        AS        0.095613  0.032932  0.053061  0.034345  0.044377  0.032401
        TM        0.087916  0.030564  0.048434  0.032779  0.044169  0.030257
        AE+AS     0.088108  0.028704  0.048712  0.031987  0.039898  0.030664
        AE+TM     0.086047  0.028237  0.047818  0.032474  0.039868  0.030371
        AS+TM     0.092044  0.031258  0.050347  0.034162  0.042667  0.031908
        AE+AS+TM  0.086052  0.027965  0.047639  0.031333  0.040318  0.030494
any     BASE      0.105211  0.025485  0.061924  0.030919  0.054662  0.034551
        AE        0.098991  0.023583  0.060206  0.029661  0.048361  0.033059
        AS        0.104881  0.026111  0.062954  0.030926  0.052958  0.034316
        TM        0.103413  0.025098  0.061803  0.030729  0.053333  0.034017
        AE+AS     0.102396  0.023687  0.061494  0.029466  0.043913  0.033937
        AE+TM     0.099245  0.023956  0.059935  0.030169  0.045887  0.033116
        AS+TM     0.103918  0.025443  0.062971  0.031250  0.052524  0.033915
        AE+AS+TM  0.100619  0.023055  0.061451  0.029516  0.044527  0.034250
sparse  BASE      0.095133  0.031038  0.056329  0.032836  0.052453  0.033912
        AE        0.090365  0.028342  0.054827  0.031605  0.043728  0.033218
        AS        0.096986  0.032398  0.057099  0.032482  0.049413  0.033960
        TM        0.094846  0.030528  0.057176  0.032730  0.048973  0.033397
        AE+AS     0.095938  0.030170  0.056541  0.030737  0.042533  0.034030
        AE+TM     0.090098  0.030128  0.054674  0.031719  0.042157  0.033169
        AS+TM     0.094609  0.031616  0.057030  0.032854  0.047492  0.033022
        AE+AS+TM  0.093627  0.027948  0.055616  0.031295  0.042450  0.033891
low     BASE      0.102867  0.024177  0.062555  0.031387  0.050112  0.034303
        AE        0.098617  0.021975  0.061416  0.030050  0.043309  0.032872
        AS        0.103791  0.024107  0.062548  0.031896  0.049489  0.034542
        TM        0.101299  0.023885  0.062466  0.031489  0.050157  0.034200
        AE+AS     0.101790  0.021740  0.062243  0.030094  0.041995  0.032625
        AE+TM     0.097165  0.021812  0.060162  0.031209  0.042928  0.032539
        AS+TM     0.103637  0.023469  0.063586  0.031846  0.048482  0.034420
        AE+AS+TM  0.099863  0.021162  0.061533  0.030263  0.043290  0.033067
medium  BASE      0.099938  0.024713  0.058353  0.029939  0.056430  0.033417
        AE        0.092716  0.022857  0.057103  0.029187  0.047635  0.032022
        AS        0.098487  0.025195  0.059851  0.030460  0.053320  0.033373
        TM        0.098351  0.024154  0.057890  0.029934  0.053784  0.033353
        AE+AS     0.095483  0.022287  0.058312  0.028896  0.043177  0.033504
        AE+TM     0.094625  0.023117  0.057427  0.030028  0.045645  0.032120
        AS+TM     0.096774  0.024543  0.059547  0.030320  0.052039  0.033396
        AE+AS+TM  0.094358  0.022386  0.058505  0.029003  0.043639  0.033769
dense   BASE      0.119544  0.023880  0.068752  0.030383  0.058358  0.036608
        AE        0.111710  0.022847  0.065802  0.028537  0.056633  0.034451
        AS        0.118100  0.024854  0.070433  0.029391  0.058093  0.035479
        TM        0.116502  0.023711  0.068462  0.029562  0.058531  0.035043
        AE+AS     0.114925  0.022760  0.067429  0.028666  0.047619  0.035787
        AE+TM     0.112070  0.022780  0.065790  0.028111  0.051295  0.034929
        AS+TM     0.117885  0.024209  0.069843  0.030720  0.059928  0.034583
        AE+AS+TM  0.112853  0.022429  0.068253  0.028146  0.048145  0.036305

# Replacing Specific Modalities

In [7]:
metrics = ['rmse_lbm', 'rmse_tgts_fys', 'rmse_tgts_onv', 'rmse_tgts_soc', 'rmse_tgts_vrz', 'rmse_tgts_won']
metric_labels = ["LIV", "PHY", "NUI", "SOC", "AME", "HOU"]

row_names = ["BASE"] + names
    
rows = []
for suffix in possible_suffixes[1:-1]:
    for name, e in zip(row_names, [""] + embeddings):
        ckpt = "ckpt_livability" + e + suffix
        pattern = f"{ckpt}/Livability_test_0320{suffix}_eval_results_test_final.txt"
        files = glob.glob(pattern)
        if len(files) == 0:
            continue
        with open(files[0]) as fp:
            result = json.load(fp)
        row = {"suffix": suffix, "model": name}
        row.update({label: result.get(m, np.nan) for m, label in zip(metrics, metric_labels)})
        rows.append(row)

all_metrics = pd.DataFrame(rows).set_index(["suffix", "model"])
save_tex(all_metrics, "tex_tables/s2_replace_modalities.tex",
         "RMSE per training modality ablation. Bold: lowest per column within each ablation group.",
         "tab:replace_modalities")
all_metrics

Saved: tex_tables/s2_replace_modalities.tex


LIV       PHY       NUI       SOC       AME       HOU
suffix    model                                                            
_NULL_RS  BASE   0.121554  0.029108  0.066832  0.036540  0.056742  0.038149
          AE     0.105409  0.024146  0.062289  0.032225  0.047253  0.034930
          AS     0.108616  0.029361  0.064222  0.033355  0.055026  0.036506
          TM     0.105525  0.026066  0.062506  0.031941  0.053412  0.034975
_NULL_DSM BASE   0.102308  0.027349  0.061086  0.031687  0.056638  0.033301
          AE     0.098741  0.024383  0.059761  0.030323  0.047219  0.033072
          AS     0.103857  0.027102  0.062283  0.032007  0.052718  0.033456
          TM     0.100012  0.026926  0.059661  0.030560  0.052061  0.032917
_NULL_GIU BASE   0.109346  0.028118  0.064817  0.033155  0.058833  0.035035
          AE     0.098155  0.024831  0.059641  0.029957  0.047704  0.033636
          AS     0.105269  0.027733  0.063680  0.031945  0.055233  0.034591
          TM     0.104776  0.026766  0.062670  0.031552  0.053190  0.034345
_noPOI    BASE   0.130579  0.028613  0.078890  0.048539  0.065731  0.042597
          AE     0.097205  0.024509  0.059576  0.029876  0.048789  0.033714
          AS     0.103726  0.027976  0.063219  0.031927  0.051930  0.034066
          TM     0.097252  0.025601  0.058892  0.029757  0.055113  0.032826

## City-level results

In [8]:
metrics = ['rmse_lbm', 'rmse_tgts_fys', 'rmse_tgts_onv', 'rmse_tgts_soc', 'rmse_tgts_vrz', 'rmse_tgts_won']
metric_labels = ["LIV", "PHY", "NUI", "SOC", "AME", "HOU"]
row_names = ["BASE"] + names

rows = []
for city, idx in city_indices.items():
    for suffix in possible_suffixes[1:-1]:
        for name, e in zip(row_names, [""] + embeddings):
            ckpt = "ckpt_livability" + e + suffix
            preds_f = glob.glob(f"{ckpt}/Livability_test_0320{suffix}_preds_test_final.npy")
            tgts_f  = glob.glob(f"{ckpt}/Livability_test_0320{suffix}_tgts_test_final.npy")
            if not preds_f or not tgts_f:
                continue
            rmse = _city_rmse(preds_f[0], tgts_f[0], idx)
            row = {"city": city, "suffix": suffix, "model": name}
            row.update({lbl: rmse[i] for i, lbl in enumerate(metric_labels)})
            rows.append(row)

df = pd.DataFrame(rows).set_index(["city", "suffix", "model"])
save_tex(df, "tex_tables/s2_city.tex",
         "City-level RMSE per training modality ablation. Bold: lowest per column within each (city, ablation) group.",
         "tab:replace_modalities_city")
df

Saved: tex_tables/s2_city.tex


LIV       PHY       NUI       SOC       AME  \
city      suffix    model                                                     
Eindhoven _NULL_RS  BASE   0.112045  0.027607  0.065452  0.034760  0.056243   
                    AE     0.093996  0.022117  0.057633  0.030033  0.043764   
                    AS     0.099562  0.028245  0.063122  0.032013  0.056300   
                    TM     0.095075  0.023819  0.059678  0.030435  0.054682   
          _NULL_DSM BASE   0.093476  0.026031  0.059711  0.029729  0.056606   
...                             ...       ...       ...       ...       ...   
Beesel    _NULL_GIU TM     0.082735  0.038522  0.044459  0.026212  0.036451   
          _noPOI    BASE   0.126736  0.025766  0.076566  0.047612  0.071085   
                    AE     0.093937  0.022660  0.058883  0.029440  0.048905   
                    AS     0.102718  0.025826  0.064753  0.030944  0.053922   
                    TM     0.095297  0.025594  0.059501  0.029295  0.058834   

                                HOU  
city      suffix    model            
Eindhoven _NULL_RS  BASE   0.031628  
                    AE     0.028888  
                    AS     0.032166  
                    TM     0.029098  
          _NULL_DSM BASE   0.028520  
...                             ...  
Beesel    _NULL_GIU TM     0.023098  
          _noPOI    BASE   0.043632  
                    AE     0.034559  
                    AS     0.034123  
                    TM     0.033158  

[64 rows x 6 columns]

## POI-density stratified results

In [9]:
metrics = ['rmse_lbm', 'rmse_tgts_fys', 'rmse_tgts_onv', 'rmse_tgts_soc', 'rmse_tgts_vrz', 'rmse_tgts_won']
metric_labels = ["LIV", "PHY", "NUI", "SOC", "AME", "HOU"]
row_names = ["BASE"] + names

rows = []
for stratum, idx in poi_strata_indices.items():
    for suffix in possible_suffixes[1:-1]:
        for name, e in zip(row_names, [""] + embeddings):
            ckpt = "ckpt_livability" + e + suffix
            preds_f = glob.glob(f"{ckpt}/Livability_test_0320{suffix}_preds_test_final.npy")
            tgts_f  = glob.glob(f"{ckpt}/Livability_test_0320{suffix}_tgts_test_final.npy")
            if not preds_f or not tgts_f:
                continue
            rmse = _city_rmse(preds_f[0], tgts_f[0], idx)
            row = {"stratum": stratum, "suffix": suffix, "model": name}
            row.update({lbl: rmse[i] for i, lbl in enumerate(metric_labels)})
            rows.append(row)

df = pd.DataFrame(rows).set_index(["stratum", "suffix", "model"])
save_tex(df, "tex_tables/s2_poi.tex",
         "POI-density stratified RMSE per training modality ablation. Bold: lowest per column within each (stratum, ablation) group.",
         "tab:replace_modalities_poi")
df

Saved: tex_tables/s2_poi.tex


LIV       PHY       NUI       SOC       AME  \
stratum suffix    model                                                     
none    _NULL_RS  BASE   0.114088  0.036307  0.055232  0.039749  0.053081   
                  AE     0.096256  0.026723  0.054276  0.034174  0.043297   
                  AS     0.093695  0.032921  0.051373  0.033727  0.048671   
                  TM     0.092521  0.030509  0.051301  0.033019  0.046629   
        _NULL_DSM BASE   0.092750  0.032242  0.050713  0.035184  0.048619   
...                           ...       ...       ...       ...       ...   
dense   _NULL_GIU TM     0.121977  0.024427  0.072585  0.031452  0.059456   
        _noPOI    BASE   0.130847  0.029206  0.078407  0.049632  0.064768   
                  AE     0.097325  0.025013  0.059581  0.029864  0.049258   
                  AS     0.103059  0.028268  0.062621  0.032273  0.052307   
                  TM     0.097463  0.026283  0.059148  0.029915  0.053939   

                              HOU  
stratum suffix    model            
none    _NULL_RS  BASE   0.036954  
                  AE     0.033362  
                  AS     0.033544  
                  TM     0.031909  
        _NULL_DSM BASE   0.030228  
...                           ...  
dense   _NULL_GIU TM     0.036349  
        _noPOI    BASE   0.042729  
                  AE     0.034059  
                  AS     0.033969  
                  TM     0.032671  

[96 rows x 6 columns]

# Zeroing out

In [10]:
metrics = ['rmse_lbm', 'rmse_tgts_fys', 'rmse_tgts_onv', 'rmse_tgts_soc', 'rmse_tgts_vrz', 'rmse_tgts_won']
metric_labels = ["LIV", "PHY", "NUI", "SOC", "AME", "HOU"]
row_names = ["BASE"] + total_names
    
rows = []
for suffix in possible_suffixes[1:]:
    for name, e in zip(row_names, [""] + total_embeddings):
        pattern = f"ckpt_livability{e}/Livability_test_0320{suffix}_eval_results_test_final.txt"
        files = glob.glob(pattern)
        if len(files) == 0:
            continue
        with open(files[0]) as fp:
            result = json.load(fp)
        
        row = {"suffix": suffix, "model": name}
        row.update({label: result.get(m, np.nan) for m, label in zip(metrics, metric_labels)})
        rows.append(row)

all_metrics = pd.DataFrame(rows).set_index(["suffix", "model"])
save_tex(all_metrics, "tex_tables/s3_zeroing_out.tex",
         "RMSE with zeroed-out inference modalities. Bold: lowest per column within each ablation group.",
         "tab:zeroing_out")
all_metrics

Saved: tex_tables/s3_zeroing_out.tex


LIV       PHY       NUI       SOC       AME       HOU
suffix    model                                                               
_NULL_RS  BASE      0.154363  0.037556  0.077735  0.049063  0.069902  0.050394
          AE        0.116266  0.025691  0.064663  0.034818  0.048342  0.036196
          AS        0.131404  0.034679  0.076046  0.045345  0.068090  0.041938
          TM        0.114838  0.030450  0.064851  0.037892  0.063598  0.037204
          AE+AS     0.113195  0.027550  0.063535  0.034299  0.044195  0.036498
          AE+TM     0.108766  0.025656  0.061502  0.032762  0.045850  0.035358
          AS+TM     0.130540  0.031399  0.074683  0.046320  0.065251  0.042980
          AE+AS+TM  0.107238  0.024472  0.062743  0.031366  0.044444  0.034952
_NULL_DSM BASE      0.105252  0.031425  0.061173  0.031711  0.060507  0.033101
          AE        0.097895  0.026620  0.058593  0.029862  0.047925  0.033167
          AS        0.103626  0.029888  0.062528  0.031321  0.057402  0.033553
          TM        0.101911  0.028608  0.060612  0.031146  0.056665  0.033691
          AE+AS     0.100564  0.026203  0.060529  0.029955  0.044773  0.033514
          AE+TM     0.096822  0.026621  0.058451  0.030503  0.045831  0.032811
          AS+TM     0.102093  0.028654  0.062399  0.031467  0.054538  0.033369
          AE+AS+TM  0.099241  0.024459  0.060160  0.029574  0.043928  0.033797
_NULL_GIU BASE      0.109236  0.039298  0.064289  0.032557  0.071338  0.034662
          AE        0.104937  0.028282  0.061126  0.032097  0.049397  0.034137
          AS        0.108785  0.035734  0.064124  0.032636  0.066507  0.034689
          TM        0.109942  0.035512  0.062991  0.032349  0.067303  0.034805
          AE+AS     0.104431  0.028380  0.062090  0.032164  0.046213  0.034996
          AE+TM     0.103168  0.028095  0.060816  0.033504  0.046931  0.033195
          AS+TM     0.108065  0.034565  0.063470  0.033654  0.060676  0.035188
          AE+AS+TM  0.103182  0.025438  0.061467  0.032299  0.044926  0.034591
_noPOI    BASE      0.096577  0.027040  0.064201  0.032857  0.070107  0.034343
          AE        0.098507  0.025740  0.061259  0.031327  0.054247  0.033928
          AS        0.099664  0.027016  0.063578  0.032611  0.067917  0.033394
          TM        0.096437  0.025909  0.062391  0.032618  0.057282  0.034111
          AE+AS     0.099989  0.025530  0.063755  0.029823  0.049416  0.033823
          AE+TM     0.094721  0.026654  0.058728  0.030948  0.050552  0.034120
          AS+TM     0.096012  0.026344  0.062996  0.031754  0.056837  0.033517
          AE+AS+TM  0.098609  0.025041  0.061085  0.029181  0.046890  0.033628

## City-level results

In [11]:
metrics = ['rmse_lbm', 'rmse_tgts_fys', 'rmse_tgts_onv', 'rmse_tgts_soc', 'rmse_tgts_vrz', 'rmse_tgts_won']
metric_labels = ["LIV", "PHY", "NUI", "SOC", "AME", "HOU"]
row_names = ["BASE"] + total_names

rows = []
for city, idx in city_indices.items():
    for suffix in possible_suffixes[1:]:
        for name, e in zip(row_names, [""] + total_embeddings):
            ckpt = "ckpt_livability" + e
            preds_f = glob.glob(f"{ckpt}/Livability_test_0320{suffix}_preds_test_final.npy")
            tgts_f  = glob.glob(f"{ckpt}/Livability_test_0320{suffix}_tgts_test_final.npy")
            if not preds_f or not tgts_f:
                continue
            rmse = _city_rmse(preds_f[0], tgts_f[0], idx)
            row = {"city": city, "suffix": suffix, "model": name}
            row.update({lbl: rmse[i] for i, lbl in enumerate(metric_labels)})
            rows.append(row)

df = pd.DataFrame(rows).set_index(["city", "suffix", "model"])
save_tex(df, "tex_tables/s3_city.tex",
         "City-level RMSE with zeroed-out inference modalities. Bold: lowest per column within each (city, ablation) group.",
         "tab:zeroing_out_city")
df

Saved: tex_tables/s3_city.tex


LIV       PHY       NUI       SOC       AME  \
city      suffix   model                                                        
Eindhoven _NULL_RS BASE      0.141111  0.032407  0.081689  0.049295  0.071892   
                   AE        0.107061  0.023524  0.062992  0.033256  0.048254   
                   AS        0.123626  0.032356  0.080236  0.043773  0.070483   
                   TM        0.102967  0.027179  0.064185  0.038907  0.066970   
                   AE+AS     0.102385  0.025951  0.061604  0.032797  0.043812   
...                               ...       ...       ...       ...       ...   
Beesel    _noPOI   TM        0.090026  0.024506  0.063063  0.030693  0.061588   
                   AE+AS     0.094275  0.023855  0.062141  0.028892  0.050577   
                   AE+TM     0.084986  0.025799  0.054196  0.030465  0.050277   
                   AS+TM     0.093169  0.023738  0.063493  0.031548  0.060942   
                   AE+AS+TM  0.093618  0.023979  0.060089  0.028462  0.046351   

                                  HOU  
city      suffix   model               
Eindhoven _NULL_RS BASE      0.042569  
                   AE        0.030821  
                   AS        0.034398  
                   TM        0.030099  
                   AE+AS     0.030923  
...                               ...  
Beesel    _noPOI   TM        0.035227  
                   AE+AS     0.034287  
                   AE+TM     0.035315  
                   AS+TM     0.033096  
                   AE+AS+TM  0.032368  

[128 rows x 6 columns]

## POI-density stratified results

In [12]:
metrics = ['rmse_lbm', 'rmse_tgts_fys', 'rmse_tgts_onv', 'rmse_tgts_soc', 'rmse_tgts_vrz', 'rmse_tgts_won']
metric_labels = ["LIV", "PHY", "NUI", "SOC", "AME", "HOU"]
row_names = ["BASE"] + total_names

rows = []
for stratum, idx in poi_strata_indices.items():
    for suffix in possible_suffixes[1:]:
        for name, e in zip(row_names, [""] + total_embeddings):
            ckpt = "ckpt_livability" + e
            preds_f = glob.glob(f"{ckpt}/Livability_test_0320{suffix}_preds_test_final.npy")
            tgts_f  = glob.glob(f"{ckpt}/Livability_test_0320{suffix}_tgts_test_final.npy")
            if not preds_f or not tgts_f:
                continue
            rmse = _city_rmse(preds_f[0], tgts_f[0], idx)
            row = {"stratum": stratum, "suffix": suffix, "model": name}
            row.update({lbl: rmse[i] for i, lbl in enumerate(metric_labels)})
            rows.append(row)

df = pd.DataFrame(rows).set_index(["stratum", "suffix", "model"])
save_tex(df, "tex_tables/s3_poi.tex",
         "POI-density stratified RMSE with zeroed-out inference modalities. Bold: lowest per column within each (stratum, ablation) group.",
         "tab:zeroing_out_poi")
df

Saved: tex_tables/s3_poi.tex


LIV       PHY       NUI       SOC       AME  \
stratum suffix   model                                                        
none    _NULL_RS BASE      0.136753  0.043650  0.061563  0.047450  0.071641   
                 AE        0.126414  0.030137  0.060585  0.036879  0.044322   
                 AS        0.105465  0.045110  0.058094  0.047663  0.072870   
                 TM        0.100783  0.035264  0.052864  0.037027  0.067537   
                 AE+AS     0.096701  0.032230  0.051216  0.034508  0.042193   
...                             ...       ...       ...       ...       ...   
dense   _noPOI   TM        0.096730  0.026537  0.061121  0.032925  0.056903   
                 AE+AS     0.100715  0.025546  0.062472  0.030017  0.049720   
                 AE+TM     0.094541  0.026705  0.057771  0.030959  0.051480   
                 AS+TM     0.096252  0.026619  0.061625  0.031733  0.055983   
                 AE+AS+TM  0.098407  0.025349  0.060503  0.029228  0.047083   

                                HOU  
stratum suffix   model               
none    _NULL_RS BASE      0.052974  
                 AE        0.037608  
                 AS        0.036856  
                 TM        0.034742  
                 AE+AS     0.033065  
...                             ...  
dense   _noPOI   TM        0.034123  
                 AE+AS     0.034083  
                 AE+TM     0.034774  
                 AS+TM     0.033924  
                 AE+AS+TM  0.033752  

[192 rows x 6 columns]

# Only Embeddings

In [13]:
metrics = ['rmse_lbm', 'rmse_tgts_fys', 'rmse_tgts_onv', 'rmse_tgts_soc', 'rmse_tgts_vrz', 'rmse_tgts_won']
metric_labels = ["LIV", "PHY", "NUI", "SOC", "AME", "HOU"]

    
rows = []
for suffix in possible_suffixes[-1:]:
    for name, e in zip(total_names, total_embeddings):
        ckpt = "ckpt_livability" + e + suffix
        pattern = f"{ckpt}/Livability_test_0320{suffix}_eval_results_test_final.txt"
        files = glob.glob(pattern)
        if len(files) == 0:
            continue
        with open(files[0]) as fp:
            result = json.load(fp)
        row = {"suffix": suffix, "model": name}
        row.update({label: result.get(m, np.nan) for m, label in zip(metrics, metric_labels)})
        rows.append(row)

all_metrics = pd.DataFrame(rows).set_index(["suffix", "model"])
save_tex(all_metrics, "tex_tables/s4_only_embeddings.tex",
         "RMSE using embeddings only (all other modalities zeroed). Bold: lowest per column.",
         "tab:only_embeddings")
all_metrics

Saved: tex_tables/s4_only_embeddings.tex


LIV       PHY       NUI       SOC       AME       HOU
suffix model                                                               
_NULL  AE        0.112173  0.024337  0.064270  0.033656  0.051344  0.035664
       AS        0.116065  0.027254  0.066720  0.034404  0.064239  0.036772
       TM        0.109677  0.025125  0.063528  0.033392  0.054845  0.034396
       AE+AS     0.113630  0.024341  0.064211  0.032962  0.048861  0.036084
       AE+TM     0.107102  0.023572  0.060954  0.032531  0.048996  0.035702
       AS+TM     0.109342  0.026267  0.066409  0.033957  0.061737  0.036394
       AE+AS+TM  0.112830  0.027717  0.064693  0.036276  0.055714  0.038596

## City-level results

In [14]:
metrics = ['rmse_lbm', 'rmse_tgts_fys', 'rmse_tgts_onv', 'rmse_tgts_soc', 'rmse_tgts_vrz', 'rmse_tgts_won']
metric_labels = ["LIV", "PHY", "NUI", "SOC", "AME", "HOU"]

rows = []
for city, idx in city_indices.items():
    for suffix in possible_suffixes[-1:]:
        for name, e in zip(total_names, total_embeddings):
            ckpt = "ckpt_livability" + e + suffix
            preds_f = glob.glob(f"{ckpt}/Livability_test_0320{suffix}_preds_test_final.npy")
            tgts_f  = glob.glob(f"{ckpt}/Livability_test_0320{suffix}_tgts_test_final.npy")
            if not preds_f or not tgts_f:
                continue
            rmse = _city_rmse(preds_f[0], tgts_f[0], idx)
            row = {"city": city, "suffix": suffix, "model": name}
            row.update({lbl: rmse[i] for i, lbl in enumerate(metric_labels)})
            rows.append(row)

df = pd.DataFrame(rows).set_index(["city", "suffix", "model"])
save_tex(df, "tex_tables/s4_city.tex",
         "City-level RMSE using embeddings only (all other modalities zeroed).",
         "tab:only_embeddings_city")
df

Saved: tex_tables/s4_city.tex


LIV       PHY       NUI       SOC       AME  \
city      suffix model                                                        
Eindhoven _NULL  AE        0.105369  0.022346  0.063250  0.031942  0.049402   
                 AS        0.112589  0.026674  0.065591  0.033382  0.062594   
                 TM        0.100744  0.024085  0.060955  0.032746  0.054691   
                 AE+AS     0.108230  0.023981  0.063292  0.032509  0.044365   
                 AE+TM     0.101250  0.021182  0.058254  0.031184  0.046675   
                 AS+TM     0.105007  0.026168  0.065049  0.033082  0.064653   
                 AE+AS+TM  0.110076  0.027873  0.060563  0.034471  0.049545   
Hengelo   _NULL  AE        0.118945  0.026390  0.060682  0.033449  0.050570   
                 AS        0.120365  0.028447  0.065297  0.034184  0.063494   
                 TM        0.126636  0.026425  0.069333  0.036269  0.045649   
                 AE+AS     0.124329  0.025493  0.060950  0.033061  0.053302   
                 AE+TM     0.120789  0.026111  0.061521  0.033172  0.044794   
                 AS+TM     0.117028  0.026836  0.067532  0.036709  0.046933   
                 AE+AS+TM  0.119462  0.028472  0.067060  0.038561  0.067765   
Dordrecht _NULL  AE        0.121203  0.025481  0.071133  0.037616  0.056967   
                 AS        0.121669  0.027060  0.072039  0.037088  0.069233   
                 TM        0.112653  0.025336  0.065231  0.032537  0.063027   
                 AE+AS     0.117785  0.023501  0.070443  0.034600  0.053976   
                 AE+TM     0.108300  0.024675  0.067478  0.035158  0.058058   
                 AS+TM     0.113307  0.025643  0.070188  0.033835  0.068491   
                 AE+AS+TM  0.114078  0.026385  0.071546  0.038327  0.056979   
Beesel    _NULL  AE        0.081055  0.028064  0.041229  0.025010  0.034342   
                 AS        0.085721  0.028772  0.044922  0.027774  0.050120   
                 TM        0.080306  0.028974  0.039792  0.028173  0.044430   
                 AE+AS     0.070756  0.027836  0.044110  0.024173  0.034477   
                 AE+TM     0.074454  0.028946  0.035201  0.024526  0.025280   
                 AS+TM     0.079835  0.028599  0.042694  0.026850  0.050292   
                 AE+AS+TM  0.093315  0.030379  0.046392  0.027846  0.033734   

                                HOU  
city      suffix model               
Eindhoven _NULL  AE        0.031445  
                 AS        0.032412  
                 TM        0.030782  
                 AE+AS     0.032536  
                 AE+TM     0.030122  
                 AS+TM     0.032873  
                 AE+AS+TM  0.036374  
Hengelo   _NULL  AE        0.041731  
                 AS        0.039184  
                 TM        0.039774  
                 AE+AS     0.041945  
                 AE+TM     0.045863  
                 AS+TM     0.041052  
                 AE+AS+TM  0.043324  
Dordrecht _NULL  AE        0.038644  
                 AS        0.042723  
                 TM        0.036746  
                 AE+AS     0.038362  
                 AE+TM     0.036604  
                 AS+TM     0.039385  
                 AE+AS+TM  0.039549  
Beesel    _NULL  AE        0.019780  
                 AS        0.027162  
                 TM        0.023056  
                 AE+AS     0.018256  
                 AE+TM     0.019558  
                 AS+TM     0.024184  
                 AE+AS+TM  0.025220

## POI-density stratified results

In [15]:
metrics = ['rmse_lbm', 'rmse_tgts_fys', 'rmse_tgts_onv', 'rmse_tgts_soc', 'rmse_tgts_vrz', 'rmse_tgts_won']
metric_labels = ["LIV", "PHY", "NUI", "SOC", "AME", "HOU"]

rows = []
for stratum, idx in poi_strata_indices.items():
    for suffix in possible_suffixes[-1:]:
        for name, e in zip(total_names, total_embeddings):
            ckpt = "ckpt_livability" + e + suffix
            preds_f = glob.glob(f"{ckpt}/Livability_test_0320{suffix}_preds_test_final.npy")
            tgts_f  = glob.glob(f"{ckpt}/Livability_test_0320{suffix}_tgts_test_final.npy")
            if not preds_f or not tgts_f:
                continue
            rmse = _city_rmse(preds_f[0], tgts_f[0], idx)
            row = {"stratum": stratum, "suffix": suffix, "model": name}
            row.update({lbl: rmse[i] for i, lbl in enumerate(metric_labels)})
            rows.append(row)

df = pd.DataFrame(rows).set_index(["stratum", "suffix", "model"])
save_tex(df, "tex_tables/s4_poi.tex",
         "POI-density stratified RMSE using embeddings only (all other modalities zeroed).",
         "tab:only_embeddings_poi")
df

Saved: tex_tables/s4_poi.tex


LIV       PHY       NUI       SOC       AME  \
stratum suffix model                                                        
none    _NULL  AE        0.097477  0.027595  0.050784  0.033094  0.042338   
               AS        0.107937  0.031675  0.054048  0.035889  0.055546   
               TM        0.098124  0.030669  0.052729  0.035502  0.052192   
               AE+AS     0.098312  0.028524  0.049658  0.034469  0.042859   
               AE+TM     0.093061  0.028254  0.047457  0.033456  0.038831   
               AS+TM     0.101711  0.031299  0.056025  0.035658  0.053554   
               AE+AS+TM  0.106154  0.032212  0.054039  0.037832  0.044807   
any     _NULL  AE        0.114172  0.023821  0.066019  0.033738  0.052537   
               AS        0.117211  0.026543  0.068384  0.034181  0.065419   
               TM        0.111272  0.024204  0.064963  0.033071  0.055224   
               AE+AS     0.115709  0.023665  0.066079  0.032735  0.049681   
               AE+TM     0.109011  0.022804  0.062692  0.032393  0.050316   
               AS+TM     0.110418  0.025444  0.067800  0.033700  0.062849   
               AE+AS+TM  0.113777  0.026994  0.066114  0.036042  0.057140   
sparse  _NULL  AE        0.104657  0.027715  0.059198  0.035499  0.046947   
               AS        0.112819  0.032099  0.060956  0.036357  0.055668   
               TM        0.109084  0.029723  0.061637  0.035675  0.052503   
               AE+AS     0.106745  0.028419  0.057863  0.035493  0.047295   
               AE+TM     0.100576  0.026707  0.055761  0.033746  0.045101   
               AS+TM     0.106224  0.030866  0.062590  0.035901  0.054299   
               AE+AS+TM  0.107665  0.032752  0.061037  0.039413  0.052118   
low     _NULL  AE        0.111150  0.021852  0.064279  0.033703  0.047326   
               AS        0.116259  0.024680  0.065224  0.034755  0.061824   
               TM        0.106523  0.023361  0.062731  0.033540  0.052773   
               AE+AS     0.113990  0.021468  0.065797  0.033822  0.046973   
               AE+TM     0.109033  0.021154  0.062088  0.032960  0.045890   
               AS+TM     0.108470  0.023638  0.065221  0.033569  0.059593   
               AE+AS+TM  0.114822  0.025562  0.065496  0.036532  0.050964   
medium  _NULL  AE        0.109932  0.022840  0.064096  0.033542  0.051427   
               AS        0.114228  0.025602  0.065458  0.033277  0.065171   
               TM        0.107336  0.022817  0.061935  0.032133  0.053948   
               AE+AS     0.111533  0.022681  0.064220  0.032011  0.047652   
               AE+TM     0.104905  0.021787  0.060751  0.032310  0.048616   
               AS+TM     0.107101  0.024415  0.065688  0.033360  0.062057   
               AE+AS+TM  0.109225  0.026056  0.065256  0.035755  0.053620   
dense   _NULL  AE        0.127818  0.024428  0.073994  0.032864  0.061909   
               AS        0.124577  0.025773  0.079103  0.033257  0.074666   
               TM        0.122199  0.022890  0.072781  0.032008  0.060879   
               AE+AS     0.127753  0.023859  0.073282  0.030548  0.056191   
               AE+TM     0.118926  0.023121  0.069618  0.030959  0.059431   
               AS+TM     0.119042  0.024815  0.075962  0.032818  0.071786   
               AE+AS+TM  0.121908  0.025603  0.070835  0.033530  0.069635   

                              HOU  
stratum suffix model               
none    _NULL  AE        0.032297  
               AS        0.036925  
               TM        0.032052  
               AE+AS     0.033406  
               AE+TM     0.031870  
               AS+TM     0.035180  
               AE+AS+TM  0.036738  
any     _NULL  AE        0.036132  
               AS        0.036750  
               TM        0.034727  
               AE+AS     0.036460  
               AE+TM     0.036231  
               AS+TM     0.036568  
               AE+AS+TM  0.038861  
sparse  _NULL  AE        0.034839  
               AS        0.03